In [1]:
import json
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

import config
from src.auditoria_sesgo import (
    cumple_regla_80, diferencia_score_por_grupo,
    razon_impacto_dispar, tasa_seleccion_por_grupo,
)
from src.db_io import leer_tabla_sqlite
from src.decisiones import decidir_interpretacion_proxy_genero
from src.features_modelo import features_modelo_a

df = leer_tabla_sqlite(config.ORO_DB, "cliente_features")
fact = leer_tabla_sqlite(config.ORO_DB, "fact_cliente_score")
datos = df.merge(fact[["numero_id", "score", "nivel", "poblacion", "modelo_usado"]],
                 on="numero_id", how="inner")
(config.OUTPUTS_DIR / "powerbi").mkdir(parents=True, exist_ok=True)
(config.OUTPUTS_DIR / "models").mkdir(parents=True, exist_ok=True)

# §6.6.1: ¿el género se filtra por variables correlacionadas?
# Si el AUC es alto, excluir desc_genero de la lista de entrada NO lo excluye del modelo.
con_genero = datos[datos["desc_genero"].notna()].reset_index(drop=True)
generos = con_genero["desc_genero"].value_counts()
print(generos.to_string())

principal = generos.index[0]
y_genero = (con_genero["desc_genero"] == principal).astype(int)
cols = features_modelo_a(con_genero.columns)
X_g = pd.get_dummies(
    con_genero[cols],
    columns=[c for c in ["desc_segmento", "grupo_edad", "desc_tipo_de_vivienda"] if c in cols],
    dummy_na=False,
)

Xg_tr, Xg_te, yg_tr, yg_te = train_test_split(
    X_g, y_genero, test_size=config.TEST_SIZE,
    random_state=config.RANDOM_STATE, stratify=y_genero)
proxy = HistGradientBoostingClassifier(random_state=config.RANDOM_STATE)
proxy.fit(Xg_tr, yg_tr)
auc_proxy = float(roc_auc_score(yg_te, proxy.predict_proba(Xg_te)[:, 1]))

sub = Xg_te.sample(n=min(30_000, len(Xg_te)), random_state=config.RANDOM_STATE)
pi = permutation_importance(proxy, sub, yg_te.loc[sub.index], n_repeats=5,
                            random_state=config.RANDOM_STATE, scoring="roc_auc")
top_proxy = (pd.DataFrame({"variable": sub.columns, "importancia": pi.importances_mean})
             .sort_values("importancia", ascending=False).head(10).reset_index(drop=True))

# D6: bandas de interpretación en vez de un umbral único — un umbral binario
# oculta el caso intermedio, que es el resultado más probable.
interpretacion_proxy = decidir_interpretacion_proxy_genero(auc_proxy)
print(f"\nAUC del clasificador de género a partir del resto de predictoras: {auc_proxy:.4f}")
print(f"Interpretación (D6): {interpretacion_proxy['interpretacion']} "
      f"-> acción: {interpretacion_proxy['accion']}")
print("\nVariables más asociadas al género:")
print(top_proxy.to_string(index=False))

with open(config.OUTPUTS_DIR / "models" / "proxy_genero.json", "w", encoding="utf-8") as f:
    json.dump({**interpretacion_proxy,
               "umbral_moderado": config.UMBRAL_AUC_PROXY_MODERADO,
               "umbral_sustancial": config.UMBRAL_AUC_PROXY_SUSTANCIAL,
               "variables_mas_asociadas": top_proxy.to_dict(orient="records")},
              f, indent=2, ensure_ascii=False)


desc_genero
femenino      472622
masculino     387430
no binario        45
trans             33



AUC del clasificador de género a partir del resto de predictoras: 0.6254
Interpretación (D6): proxy moderado -> acción: documentar_variables_asociadas

Variables más asociadas al género:
                   variable  importancia
saldo_invertido_no_etiqueta     0.019030
          estimador_ingreso     0.013891
         ingresos_mensuales     0.013879
             grupo_edad_65+     0.013173
    total_egresos_mensuales     0.009617
    bolsillos_saldo_prom_6m     0.005818
           grupo_edad_50-65     0.005118
   bolsillos_saldo_snapshot     0.004627
           grupo_edad_18-25     0.004207
              total_activos     0.003119


In [2]:
ATRIBUTOS = ["desc_genero", "grupo_edad", "desc_tipo_de_vivienda"]

filas = []
for atributo in ATRIBUTOS:
    tasas = tasa_seleccion_por_grupo(datos, atributo, "nivel", nivel_objetivo="A")
    razon = razon_impacto_dispar(tasas.set_index("grupo")["tasa_seleccion"])
    cumple = cumple_regla_80(razon, umbral=config.UMBRAL_IMPACTO_DISPAR)
    scores = diferencia_score_por_grupo(datos, atributo, "score")

    t = tasas.merge(scores[["grupo", "score_medio", "score_mediano", "p_valor_vs_resto"]],
                    on="grupo", how="left")
    t.insert(0, "atributo", atributo)
    t["razon_impacto_dispar"] = razon
    t["cumple_regla_80"] = cumple
    filas.append(t)

    estado = "OK" if cumple else "HALLAZGO — por debajo de 0.8"
    print(f"\n=== {atributo} — razón de impacto dispar = {razon:.3f} [{estado}] ===")
    print(t[["grupo", "n", "tasa_seleccion", "score_medio", "p_valor_vs_resto"]]
          .to_string(index=False))

auditoria = pd.concat(filas, ignore_index=True)
auditoria["auc_proxy_genero"] = auc_proxy
auditoria["interpretacion_proxy_genero"] = interpretacion_proxy["interpretacion"]
auditoria.rename(columns={"tasa_seleccion": "tasa_seleccion_nivel_A"}, inplace=True)
auditoria.to_csv(config.OUTPUTS_DIR / "powerbi" / "fact_auditoria_sesgo.csv", index=False)



=== desc_genero — razón de impacto dispar = 0.000 [HALLAZGO — por debajo de 0.8] ===
     grupo      n  tasa_seleccion  score_medio  p_valor_vs_resto
  Sin dato     93        0.000000     0.006597      2.360504e-28
  femenino 472622        0.266604     0.086912     1.007531e-264
 masculino 387430        0.229838     0.072792     1.827538e-259
no binario     45        0.111111     0.025342      6.610058e-06
     trans     33        0.090909     0.017504      1.023541e-05



=== grupo_edad — razón de impacto dispar = 0.328 [HALLAZGO — por debajo de 0.8] ===
grupo      n  tasa_seleccion  score_medio  p_valor_vs_resto
18-25 110400        0.096024     0.035511      0.000000e+00
26-35 212490        0.243282     0.082884      0.000000e+00
36-49 236761        0.293042     0.103392      0.000000e+00
50-65 163234        0.275996     0.092117     4.528669e-286
  65+ 137338        0.279078     0.059943      0.000000e+00



=== desc_tipo_de_vivienda — razón de impacto dispar = 0.519 [HALLAZGO — por debajo de 0.8] ===
     grupo      n  tasa_seleccion  score_medio  p_valor_vs_resto
 ARRENDADA  45949        0.265642     0.085635      7.817134e-46
  FAMILIAR 129359        0.335307     0.119008      0.000000e+00
NO INFORMA  18190        0.316712     0.101881     7.220915e-160
    PROPIA  75034        0.402151     0.135531      0.000000e+00
  Sin dato 591691        0.208791     0.064102      0.000000e+00


In [3]:
# SPEC_V2 §6.4: verificar que el modelo no excluya sistemáticamente a los grupos
# de mayor edad. La adopción digital correlaciona con edad y existe el riesgo de
# estar prediciendo "usa aplicaciones" en vez de "quiere invertir".
edad = auditoria[auditoria["atributo"] == "grupo_edad"].sort_values("grupo")
print("Tasa de selección al nivel A por grupo de edad:")
print(edad[["grupo", "n", "tasa_seleccion_nivel_A", "score_medio"]].to_string(index=False))

peor = edad.loc[edad["tasa_seleccion_nivel_A"].idxmin()]
mejor = edad.loc[edad["tasa_seleccion_nivel_A"].idxmax()]
print(f"\nGrupo menos favorecido: {peor['grupo']} ({peor['tasa_seleccion_nivel_A']:.2%})")
print(f"Grupo más favorecido:  {mejor['grupo']} ({mejor['tasa_seleccion_nivel_A']:.2%})")
print(f"Razón: {edad['razon_impacto_dispar'].iloc[0]:.3f}")

print(
    "\nSPEC_V2 §6.6 — `desc_genero` se conserva en dim_cliente EXCLUSIVAMENTE para "
    "esta auditoría y para caracterización descriptiva del tablero. Nunca como "
    "predictora (verificado por src/features_modelo.py y tests/test_features_modelo.py). "
    "La exclusión es un criterio de idoneidad (SPEC_V2 §6.4), no de poder predictivo: "
    "si `desc_genero` resultara predictivo, eso reflejaría una desigualdad histórica de "
    "acceso, no una señal legítima que el modelo deba aprender a usar."
)


Tasa de selección al nivel A por grupo de edad:
grupo      n  tasa_seleccion_nivel_A  score_medio
18-25 110400                0.096024     0.035511
26-35 212490                0.243282     0.082884
36-49 236761                0.293042     0.103392
50-65 163234                0.275996     0.092117
  65+ 137338                0.279078     0.059943

Grupo menos favorecido: 18-25 (9.60%)
Grupo más favorecido:  36-49 (29.30%)
Razón: 0.328

SPEC_V2 §6.6 — `desc_genero` se conserva en dim_cliente EXCLUSIVAMENTE para esta auditoría y para caracterización descriptiva del tablero. Nunca como predictora (verificado por src/features_modelo.py y tests/test_features_modelo.py). La exclusión es un criterio de idoneidad (SPEC_V2 §6.4), no de poder predictivo: si `desc_genero` resultara predictivo, eso reflejaría una desigualdad histórica de acceso, no una señal legítima que el modelo deba aprender a usar.
